# Fuzzy Logic & K-Medoids Integration

In this notebook, we demonstrate how our Fuzzy Logic system is explicitly tied to our K-Medoids clustering output. As requested by the TA, we use the Fuzzy system as an Expert Decision System.

**Inputs:**
1. `Age`
2. `K-Medoids Cluster Label` (0 = Hot Leads, 1 = Warm Leads, 2 = Cold Leads)

**Output:**
`Action Score` (Used to decide if we TARGET, MONITOR, or IGNORE the lead).

In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# 1. Define Antecedents (Inputs) and Consequent (Output)
age_var = ctrl.Antecedent(np.arange(20, 85, 1), 'age')
cluster_var = ctrl.Antecedent(np.arange(0, 3, 1), 'cluster_label')
action_score = ctrl.Consequent(np.arange(0, 11, 1), 'action_score')


## Define Membership Functions (Ranges)

In [ ]:
# Age ranges
age_var['young'] = fuzz.trimf(age_var.universe, [20, 20, 35])
age_var['middle'] = fuzz.trimf(age_var.universe, [30, 45, 60])
age_var['senior'] = fuzz.trimf(age_var.universe, [55, 85, 85])

# K-Medoids Cluster outputs
cluster_var['hot_lead'] = fuzz.trimf(cluster_var.universe, [0, 0, 1])
cluster_var['warm_lead'] = fuzz.trimf(cluster_var.universe, [0, 1, 2])
cluster_var['cold_lead'] = fuzz.trimf(cluster_var.universe, [1, 2, 2])

# Output Decision Action Ranges
action_score.automf(names=['ignore', 'monitor', 'target'])


## Define Expert Rules (Integrating K-Medoids)

In [ ]:
rules = [
    ctrl.Rule(age_var['young'] & cluster_var['cold_lead'], action_score['ignore']),
    ctrl.Rule(age_var['middle'] & cluster_var['hot_lead'], action_score['target']),
    ctrl.Rule(age_var['senior'] & cluster_var['hot_lead'], action_score['monitor']),
    ctrl.Rule(cluster_var['cold_lead'], action_score['ignore']),
    ctrl.Rule(cluster_var['warm_lead'], action_score['monitor'])
]

fuzzy_ctrl = ctrl.ControlSystem(rules)
fuzzy_sim = ctrl.ControlSystemSimulation(fuzzy_ctrl)


## Test the System (Simulating K-Medoids Pipeline output)

In [ ]:
# Example: A customer who is 45 years old and was assigned to Cluster 0 (Hot Lead) by K-Medoids.
fuzzy_sim.input['age'] = 45
fuzzy_sim.input['cluster_label'] = 0  # <--- Linked exactly to K-Medoids Output!

fuzzy_sim.compute()
score = round(fuzzy_sim.output['action_score'], 1)

print(f"Fuzzy Output Score: {score} / 10")

if score >= 6.5:
    print("Final Business Decision: TARGET (Shortlisted)")
elif score >= 4.0:
    print("Final Business Decision: MONITOR (Waitlisted)")
else:
    print("Final Business Decision: IGNORE (Rejected)")
